# 02b · EDA — ¿qué hay realmente en los datos para predecir quién pierde el curso (nota < 70)?

> Continúa `01b_caso_negocio_aprueba_o_no.ipynb`. Allí quedó definido: **el estudiante aprueba con 70 o más (`Pass`) y pierde el curso con menos de 70 (`Fail`; 69 pierde)**; clase de interés **`Fail`** (**69,2 %**);
> costos 5 a 1 (supuesto); la estrategia sin modelo a vencer es **«avisar a todos»** (costo **0,31** por estudiante); un modelo debe lograr un **NPV ≥ 83,3 %**.
> Dataset: `data/student_exam_performance.csv` (100.000 × 44). Diccionario: `data/README.md`. Plantilla: `notebooks/02_eda.ipynb`.

> **El objetivo se deriva de `exam_score` con el corte en 70** (`add_target`, de `src/proyecto/data/cleaning.py`, la misma función que usa la limpieza).
> La columna `pass_status` del dataset usa el corte en **50**: **no es el objetivo** y aquí se trata como una columna de fuga.

El EDA **no es hacer gráficos**: es descubrir qué decisiones hay que tomar antes de modelar. Este notebook termina con una **tabla de decisiones**
(cada una con cifra, acción y motivo), que también va a `reports/01_eda.md`.

**Preguntas guía:** (1) unidad de análisis · (2) objetivo · (3) suciedad · (4) qué debería ser estable · (5) sospecha de fuga.
Añadimos dos preguntas propias del caso: (6) **qué variables se relacionan con perder el curso** y (7) **si el modelo podría tratar distinto a algún grupo**.

> **Aviso metodológico:** el EDA mira *todos* los datos para entender y orientar. **No decide por sí solo qué variables entran al modelo**: eso se mide con validación cruzada en la etapa 4.
> Tampoco abre una partición de prueba: aún no existe (se crea en la etapa 3).

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from proyecto.data.cleaning import PASS_THRESHOLD, TARGET, add_target   # la misma regla de 70 que usa la limpieza

try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:                      # el EDA sigue funcionando con tablas
    HAS_PLT = False
    print("matplotlib no está instalado: se muestran tablas sin gráficos (uv pip install -r requirements.txt).")

## 0. Cargar los datos

In [ ]:
CANDIDATES = [Path.cwd().resolve().parent / "data" / "raw" / "student_exam_performance.csv",
              Path.cwd().resolve().parent / "data" / "student_exam_performance.csv",
              Path("data/raw/student_exam_performance.csv"), Path("data/student_exam_performance.csv")]
path = next((p for p in CANDIDATES if p.exists()), None)
assert path is not None, "No encontré student_exam_performance.csv en data/raw/ ni en data/"

df = pd.read_csv(path, low_memory=False)
print(f"{path.name}: {df.shape[0]:,} filas x {df.shape[1]} columnas")

df = add_target(df)        # crea pass_70: "Pass" si exam_score >= 70, "Fail" si < 70 (69 y 69,99 pierden)
ID_COL = "student_id"      # identifica al estudiante (seudónimo; no es un dato personal directo)
print(f"objetivo: {TARGET} (corte en {PASS_THRESHOLD}); la columna pass_status del dataset usa el corte de 50 y NO es el objetivo")

# Grupos de columnas definidos en el caso de negocio (01b) y en data/README.md
LEAK_COLS = ["exam_score", "pass_status", "performance_grade", "performance_level", "questions_correct", "questions_attempted"]
REVIEW_COLS = ["time_management_score", "exam_anxiety_level", "exam_difficulty"]         # a medir con y sin (etapa 4)
SENSITIVE_COLS = ["age", "gender", "family_income", "parent_education", "urban_rural", "school_type"]

fail = (df[TARGET] == "Fail")                       # True si pierde el curso (nota < 70)
num_cols = df.select_dtypes("number").columns.tolist()
cat_cols = [c for c in df.select_dtypes("object").columns if c not in [ID_COL, TARGET]]
print(f"{len(num_cols)} numéricas · {len(cat_cols)} categóricas (sin id ni objetivo; incluyen pass_status, performance_grade y performance_level)")
# Datos personales: no hay nombres, documentos ni contactos. No se imprime ninguna fila con id ni atributos de un estudiante.

## 1. ¿Cuál es la unidad de análisis? ¿Qué representa una fila?

*Pregunta:* ¿una fila es un cliente, una transacción...? ¿Alguna entidad aparece en **varias filas**? De eso depende cómo se parte el dato.

In [ ]:
print("filas:", len(df), "| columnas:", df.shape[1])
print("estudiantes distintos:", df[ID_COL].nunique())
print("filas por estudiante:", df.groupby(ID_COL).size().value_counts().to_dict())
print("filas duplicadas (sin el id ni el objetivo):", df.drop(columns=[ID_COL, TARGET]).duplicated().sum())

**Mi respuesta (una frase):** una fila es **un estudiante** (14 a 20 años) con sus hábitos, su contexto y el resultado de un examen; `student_id` es único (100.000 ids para 100.000 filas), así que **ningún estudiante se repite**.

**Unidad por la que partiré los datos:** **por fila, estratificado por `pass_70`**. No hace falta partir por grupo, porque no hay entidades repetidas; estratificar mantiene el 69,2 % de `Fail` en entrenamiento y prueba.

## 2. ¿Cómo está el objetivo?

*Pregunta:* ¿qué porcentaje acierta predecir siempre la clase más frecuente? ¿El objetivo tiene vacíos? ¿Qué tan cerca del corte (70) está la gente? ¿En qué se diferencia del `pass_status` del dataset?

In [ ]:
dist = df[TARGET].value_counts()
display(pd.DataFrame({"estudiantes": dist, "proporción": (dist / len(df)).round(4)}))
print("vacíos en el objetivo:", df[TARGET].isna().sum(), "| vacíos en exam_score:", df["exam_score"].isna().sum())
print(f"¿Pass ⇔ exam_score ≥ {PASS_THRESHOLD} en todas las filas?", bool(((df.exam_score >= PASS_THRESHOLD) == (df[TARGET] == "Pass")).all()))
print(f"Notas entre 69 y 70 (pierden el curso por menos de 1 punto): {int(((df.exam_score >= 69) & (df.exam_score < 70)).sum()):,} "
      f"| notas exactamente 70 (aprueban): {int((df.exam_score == 70).sum()):,}")

print("\n¿En qué se diferencia del pass_status del dataset (corte en 50)?")
display(pd.crosstab(df.pass_status.rename("pass_status (corte 50)"), df[TARGET].rename("pass_70 (corte 70)")))

# Línea base: las tres estrategias sin modelo, con los costos del supuesto de 01b (falso negativo = 5, falso positivo = 1)
FN_COST, FP_COST = 5, 1
tasa_fail = fail.mean()
base = pd.Series({"nunca avisar (siempre Pass)": tasa_fail * FN_COST,
                  "avisar a todos (siempre Fail)": (1 - tasa_fail) * FP_COST,
                  "azar según proporciones": tasa_fail * (1 - tasa_fail) * (FN_COST + FP_COST)},
                 name="costo esperado por estudiante").round(2)
display(base)
print(f"Exactitud de 'siempre Fail': {tasa_fail:.1%}  |  de 'siempre Pass': {1 - tasa_fail:.1%}")
print(f"NPV mínimo para vencer a «avisar a todos»: {FN_COST / (FN_COST + FP_COST):.1%}")

cerca = (df.exam_score - PASS_THRESHOLD).abs() < 5
print(f"\nA menos de 5 puntos del corte ({PASS_THRESHOLD}): {cerca.mean():.1%} de los estudiantes; de ellos pierde el curso {fail[cerca].mean():.1%}")

if HAS_PLT:
    fig, ax = plt.subplots(figsize=(7, 3))
    df.exam_score.hist(bins=50, ax=ax, color="#7f9bb5")
    ax.axvline(PASS_THRESHOLD, color="crimson", linestyle="--", label=f"corte: {PASS_THRESHOLD}")
    ax.set_title("exam_score (solo para describir el objetivo; NO es variable de entrada)")
    ax.legend(); plt.tight_layout(); plt.show()

**Hallazgo:** `Fail` (nota < 70) **69,2 %** (69.210 estudiantes) y `Pass` 30,8 % (30.790), sin vacíos en el objetivo. La regla `Pass ⇔ exam_score ≥ 70` se cumple en las 100.000 filas. Hay **2.224 estudiantes con una nota entre 69 y 70** que pierden el curso por menos de un punto, y 24 con exactamente 70 que aprueban.
**Todos** los que reprueban con el corte de 50 (22.611) también reprueban con 70, pero **46.599 aprobarían con 50 y pierden el curso con 70**. **21,8 % de los estudiantes está a menos de 5 puntos del corte** y de ellos reprueba 53,7 %: casi cara o cruz.

**Línea base (qué resultado obtiene una regla trivial):** `Fail` es ahora la clase mayoritaria. «Avisar a todos» acierta 69,2 % y cuesta **0,31** por estudiante: **es la estrategia a vencer**. «Nunca avisar» acierta 30,8 % y cuesta **3,46**; el azar, 1,28. Un modelo solo supera a «avisar a todos» si entre sus «tranquilos» al menos **83,3 % aprueba (NPV)**.

## 3. ¿Qué tan sucios están los datos y de qué manera?

*Preguntas:* ¿números guardados como texto? ¿valores centinela? ¿valores imposibles (saltos en los percentiles, negativos)? ¿vacíos? ¿duplicados? ¿categorías raras? ¿datos personales?

In [ ]:
print("Tipos:", df.dtypes.value_counts().to_dict())

# 3.1 Números guardados como texto
como_texto = {c: float(pd.to_numeric(df[c], errors="coerce").notna().mean()) for c in cat_cols + [ID_COL]
              if pd.to_numeric(df[c], errors="coerce").notna().any()}
print("3.1 columnas de texto que en realidad son números:", como_texto or "ninguna")

# 3.2 Valores centinela (-1, 999, 9999, '?', 'None'...)
sentinel_nums = {c: int(df[c].isin([-1, -999, 999, 9999]).sum()) for c in num_cols if df[c].isin([-1, -999, 999, 9999]).any()}
sentinel_txt = {c: int(df[c].isin(["?", "None", "NA", "N/A", "nan", "", "-", "unknown", "Unknown"]).sum()) for c in cat_cols
                if df[c].isin(["?", "None", "NA", "N/A", "nan", "", "-", "unknown", "Unknown"]).any()}
print("3.2 centinelas numéricos:", sentinel_nums or "ninguno", "| de texto:", sentinel_txt or "ninguno")
print("    negativos:", {c: int((df[c] < 0).sum()) for c in num_cols if (df[c] < 0).any()} or "ninguno")

# 3.2b Valores pegados al máximo o al mínimo (variables truncadas)
cont = [c for c in num_cols if df[c].nunique() > 2]
tope = pd.DataFrame({"% en el máximo": [(df[c] == df[c].max()).mean() * 100 for c in cont],
                     "máximo": [df[c].max() for c in cont],
                     "% en el mínimo": [(df[c] == df[c].min()).mean() * 100 for c in cont]}, index=cont)
print("\n3.2b variables con más de 2 % de los valores pegados al máximo o al mínimo:")
display(tope[(tope["% en el máximo"] >= 2) | (tope["% en el mínimo"] >= 2)].round(1).sort_values("% en el máximo", ascending=False))

In [ ]:
# 3.3 Valores imposibles: saltos en los percentiles (de p99 al máximo) y rangos lógicos
pct = df[cont].quantile([.5, .9, .95, .97, .99, 1]).T
pct.columns = ["mediana", "p90", "p95", "p97", "p99", "máximo"]
pct["máx / p99"] = (pct["máximo"] / pct["p99"]).round(2)
print("Percentiles (un 'máx / p99' muy alto sugiere un valor imposible):")
display(pct.round(2).sort_values("máx / p99", ascending=False).head(8))

reglas = {
    "previous_exam_score fuera de 0–100": ((df.previous_exam_score < 0) | (df.previous_exam_score > 100)).sum(),
    "attendance_percentage fuera de 0–100": ((df.attendance_percentage < 0) | (df.attendance_percentage > 100)).sum(),
    "assignment_completion_rate fuera de 0–100": ((df.assignment_completion_rate < 0) | (df.assignment_completion_rate > 100)).sum(),
    "exam_score fuera de 0–100": ((df.exam_score < 0) | (df.exam_score > 100)).sum(),
    "previous_gpa fuera de 1–4": ((df.previous_gpa < 1) | (df.previous_gpa > 4)).sum(),
    "sleep_hours fuera de 0–24": ((df.sleep_hours < 0) | (df.sleep_hours > 24)).sum(),
    "age fuera de 14–20": ((df.age < 14) | (df.age > 20)).sum(),
    "questions_correct > questions_attempted": (df.questions_correct > df.questions_attempted).sum(),
}
print("\nRangos lógicos (cuántas filas los violan):")
display(pd.Series({k: int(v) for k, v in reglas.items()}, name="filas"))

In [ ]:
# 3.5 Vacíos: cuántos, y si el vacío anticipa el resultado
filas = []
for c in df.columns[df.isna().any()]:
    m = df[c].isna()
    filas.append({"columna": c, "vacíos": int(m.sum()), "% vacíos": round(m.mean() * 100, 1),
                  "% Fail con vacío": round(fail[m].mean() * 100, 1), "% Fail sin vacío": round(fail[~m].mean() * 100, 1)})
display(pd.DataFrame(filas).sort_values("% vacíos", ascending=False).reset_index(drop=True))

n_vacios_fila = df.isna().sum(axis=1)
print(f"Filas con al menos 1 vacío: {(n_vacios_fila >= 1).sum():,} ({(n_vacios_fila >= 1).mean():.1%})  |  con 2 o más: {(n_vacios_fila >= 2).mean():.1%}")
print("Vacíos por fila:", n_vacios_fila.value_counts().sort_index().to_dict())
print(f"→ Borrar las filas con algún vacío eliminaría {(n_vacios_fila >= 1).sum():,} estudiantes.")

# 3.8 Categorías raras (< 5 %)
raras = {c: df[c].value_counts(normalize=True)[lambda s: s < 0.05].round(3).to_dict() for c in cat_cols}
print("\n3.8 categorías con menos de 5 % de las filas:", {k: v for k, v in raras.items() if v})

**Hallazgos de calidad (con cifras):**

- **Es un dataset bastante limpio**: sin números guardados como texto, sin centinelas (`-1`, `999`…), sin negativos, sin duplicados y con **0 filas fuera de los rangos lógicos** (porcentajes 0–100, `previous_gpa` 1–4, `sleep_hours` 0–24, `age` 14–20, `questions_correct` ≤ `questions_attempted`).
- **Vacíos en 7 columnas** (3,0 %–9,9 %): `attendance_percentage` 9,9 %, `time_management_score` 9,6 %, `notes_quality` 8,4 %, `previous_gpa` 7,8 %, `sleep_quality` 7,0 %, `parent_education` 6,5 %, `device_availability` 3,0 %. **41,9 % de las filas (41.940) tiene al menos un vacío**, así que borrarlas sería perder casi la mitad de los estudiantes.
- **El vacío no anticipa el resultado**: `Fail` es 68,3 %–69,4 % con vacío frente a ≈ 69,2 % sin vacío.
- **Variables truncadas en el máximo**: `stress_level` tiene 69,9 % de los valores en 10 y `exam_anxiety_level` 38,6 %; `time_management_score` 7,4 % en 100 y `attendance_percentage` 2,7 % en 100. No son errores, pero esas variables varían poco o están «cortadas» arriba.
- **Extremos altos pero posibles**: `study_hours_per_day` llega a 12 (p99 = 8,4) y `online_learning_hours` a 15 (p99 = 9,2); no hay un salto imposible.
- **Categoría rara:** solo `gender` = `Other` (2,0 %, 2.029 estudiantes).
- **Datos personales:** no hay nombres, documentos ni contactos. `student_id` es un seudónimo; `age` incluye menores (14–17).

## 4. ¿Qué variables deberían ser estables y no lo son?

*La plantilla pregunta por variables que cambian dentro de una misma entidad.* **No aplica**: cada estudiante aparece en **una sola fila**, así que no hay nada que comparar entre filas.
En su lugar se comprueba la **coherencia interna** entre columnas que deberían cuadrar.

In [ ]:
ratio = df.self_study_hours / df.study_hours_per_day
print("self_study_hours ≤ study_hours_per_day en todas las filas:", bool((df.self_study_hours <= df.study_hours_per_day).all()),
      f"| razón self/study: mín {ratio.min():.2f}, mediana {ratio.median():.2f}, máx {ratio.max():.2f}")
print("study_hours_per_day + online_learning_hours > 24 h:", int(((df.study_hours_per_day + df.online_learning_hours) > 24).sum()),
      f"(máximo {(df.study_hours_per_day + df.online_learning_hours).max():.1f} h)")
suma_triple = df.study_hours_per_day + df.self_study_hours + df.online_learning_hours
print("Si se SUMARAN las tres columnas de horas, filas con más de 24 h:", int((suma_triple > 24).sum()))
print("correlación exam_score ~ questions_correct:", round(df.exam_score.corr(df.questions_correct), 3),
      "| ~ questions_correct/questions_attempted:", round(df.exam_score.corr(df.questions_correct / df.questions_attempted), 3))

**Hallazgo:** `self_study_hours` es **menor o igual** que `study_hours_per_day` en el 100 % de las filas (entre 0,49 y 0,91 de ese valor): parece **una parte** de las horas de estudio, no un tiempo aparte. Sumar `study_hours_per_day + self_study_hours + online_learning_hours` **duplicaría horas** y daría 63 estudiantes con más de 24 h.

**¿Ruido o cambio real? ¿Por qué?** Es una **relación estructural**, no ruido. Consecuencia: la idea de variable `total_study_hours` (la suma de las tres) **se descarta**: no se suma `self_study_hours`; si se quiere una variable nueva, se usa la **proporción** `self_study_hours / study_hours_per_day`.
Además, `questions_correct` es casi la nota (correlación 0,975), lo que confirma la fuga.

## 5. ¿Hay algo que suene a fuga de información?

*Pregunta:* ¿alguna columna contiene la respuesta, o solo se conoce **después** de la decisión (antes del examen)? Lista las sospechosas; **las dudosas no se descartan por intuición**, se miden en la etapa 4.

In [ ]:
def auc_fail(x, is_fail):
    """AUC de una sola variable para separar Fail de Pass: 0,5 = sin señal; lejos de 0,5 (en cualquier sentido) = hay señal."""
    s = pd.Series(np.asarray(x, dtype=float))
    ok = s.notna().to_numpy()
    ranks = s[ok].rank().to_numpy()
    f = np.asarray(is_fail, dtype=bool)[ok]
    n1, n0 = f.sum(), (~f).sum()
    return (ranks[f].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

# ¿Qué tanto de la respuesta (pass_70) dan las columnas derivadas de la nota?
for col in ["pass_status", "performance_grade", "performance_level"]:
    tabla = pd.crosstab(df[col], df[TARGET])
    mixtas = tabla[(tabla["Fail"] > 0) & (tabla["Pass"] > 0)].sum().sum()
    print(f"\n{col} vs {TARGET}  —  da la respuesta exacta al {1 - mixtas / len(df):.1%} de los estudiantes:")
    display(tabla)

sospechosas = ["exam_score", "questions_correct", "questions_attempted"] + [c for c in REVIEW_COLS if c != "exam_difficulty"]
tabla = pd.DataFrame({"AUC como señal de Fail": {c: round(auc_fail(df[c], fail), 3) for c in sospechosas}})
tabla["qué es"] = ["La nota: de ella sale el objetivo", "Resultado del examen", "Dato del examen",
                   "Se mide en torno al examen (a medir con y sin)", "Se mide en torno al examen (a medir con y sin)"]
display(tabla)
print("exam_difficulty (se conoce al elegir el examen, a confirmar): % de Fail por nivel")
display((df.groupby("exam_difficulty")[TARGET].apply(lambda s: (s == "Fail").mean()) * 100).round(1))

**Columnas sospechosas de fuga y por qué:**

| Columna | Evidencia | Decisión |
|---|---|---|
| `exam_score` | AUC = 0,00 (separa perfecto): el objetivo es `exam_score` < 70 | **Excluir** (después de crear el objetivo) |
| `pass_status` | Es la misma respuesta con **otro corte (50)**: los 22.611 `Fail` con 50 son siempre `Fail` con 70. **No es el objetivo** | **Excluir** |
| `performance_grade` | `A` y `B` son siempre `Pass`; `D` y `F`, siempre `Fail`; solo `C` (21.794) mezcla (11.713 `Fail` y 10.081 `Pass`). Da la respuesta exacta a **78,2 %** de los estudiantes | **Excluir** |
| `performance_level` | `High` es siempre `Pass` y `Low` siempre `Fail`; `Medium` (41.999) mezcla. Da la respuesta exacta a **58,0 %** | **Excluir** |
| `questions_correct` | AUC = 0,02 y correlación 0,975 con la nota: es el resultado del examen | **Excluir** |
| `questions_attempted` | AUC ≈ 0,50, pero es un dato del propio examen: no existe antes de rendirlo | **Excluir** |
| `time_management_score`, `exam_anxiety_level` | AUC ≈ 0,45 y 0,55: poca señal; se miden en torno al examen | **A medir con y sin** (etapa 4) |
| `exam_difficulty` | `Fail`: 51,1 % (Easy), 70,2 % (Medium), 85,1 % (Hard): la señal más fuerte entre las categóricas; debería conocerse al elegir el examen | **Confirmar si se conoce antes**; medir con y sin |

## 6. ¿Qué variables se relacionan con perder el curso?

*Pregunta propia del caso:* con lo que se sabe **antes del examen**, ¿qué variables separan a los que pierden el curso? Es orientativo: **la selección final se mide con validación cruzada (etapa 4)**.

In [ ]:
predictoras_num = [c for c in num_cols if c not in LEAK_COLS]
auc_tab = pd.DataFrame({"AUC": {c: auc_fail(df[c], fail) for c in predictoras_num}})
auc_tab["fuerza (|AUC−0,5|+0,5)"] = ((auc_tab.AUC - 0.5).abs() + 0.5).round(3)
auc_tab["sentido"] = np.where(auc_tab.AUC < 0.5, "valores bajos → más Fail", "valores altos → más Fail")
auc_tab = auc_tab.sort_values("fuerza (|AUC−0,5|+0,5)", ascending=False)
print("Variables numéricas (sin las columnas de fuga), ordenadas por fuerza de la señal:")
display(auc_tab[["fuerza (|AUC−0,5|+0,5)", "sentido"]])

def fail_by_quintile(col):
    q = pd.qcut(df[col], 5, duplicates="drop")
    return (fail.groupby(q).mean() * 100).round(1)

for col in ["previous_exam_score", "previous_gpa", "study_hours_per_day"]:
    print(f"\n% de Fail por quintil de {col} (del más bajo al más alto):")
    display(fail_by_quintile(col).to_frame("% Fail"))

# Variables categóricas: % de Fail por nivel y diferencia entre el nivel más alto y el más bajo
cats = [c for c in cat_cols if c not in LEAK_COLS]
rangos = {}
for c in cats:
    g = (fail.groupby(df[c].fillna("(vacío)")).mean() * 100)
    rangos[c] = {"mayor % Fail": g.idxmax(), "menor % Fail": g.idxmin(), "diferencia (puntos)": round(g.max() - g.min(), 1)}
print("\nVariables categóricas, ordenadas por la diferencia de % de Fail entre sus niveles:")
display(pd.DataFrame(rangos).T.sort_values("diferencia (puntos)", ascending=False))

if HAS_PLT:
    fig, ax = plt.subplots(figsize=(7, 4))
    auc_tab["fuerza (|AUC−0,5|+0,5)"].sort_values().plot.barh(ax=ax, color="#7f9bb5")
    ax.axvline(0.5, color="gray", linestyle=":"); ax.set_xlim(0.45, max(0.8, auc_tab["fuerza (|AUC−0,5|+0,5)"].max() + 0.02))
    ax.set_title("Señal de cada variable numérica para separar Fail (0,5 = nada)")
    plt.tight_layout(); plt.show()

**Lo que dicen los datos:**

- **Historial académico, la señal más fuerte:** por quintiles de `previous_exam_score` (del más bajo al más alto), `Fail` pasa de **90,7 %** a **41,7 %**; con `previous_gpa`, de **89,6 %** a **43,1 %**. **Ni siquiera el quintil con mejor historial se salva**: más de 4 de cada 10 pierden el curso con el corte en 70.
- **Esfuerzo de estudio, señal media:** `study_hours_per_day` (`Fail` de 84,1 % a 47,1 %), `self_study_hours`, `practice_tests_completed` y `exam_preparation_days`.
- **Dificultad del examen:** `Fail` es 51,1 % si es fácil y **85,1 % si es difícil**, la mayor diferencia entre las categóricas (33,9 puntos).
- **Muchas variables casi no aportan:** `age`, `internet_access`, `private_tuition`, `daily_screen_time`, `physical_activity_hours`, `online_learning_hours` y `online_course_hours` tienen una señal ≈ 0,50 (nada), y las categóricas de hábitos (`study_method`, `motivation_level`, `study_environment`…) cambian el % de `Fail` en menos de 1,3 puntos.
- **Qué significa para el negocio:** el modelo se apoyará sobre todo en **notas previas, horas de estudio y dificultad**. Los «hábitos» que la institución podría querer trabajar (sueño, motivación, método de estudio) casi no separan a quien pierde el curso en estos datos. Eso es una asociación en un dataset que parece sintético, **no una conclusión causal**.

## 7. ¿Podría el modelo tratar distinto a algún grupo?

*Pregunta propia del caso:* los atributos sensibles, ¿se relacionan con el resultado? Si es así, un modelo puede aprender esa diferencia (directamente o por variables parecidas) y fallar más con algún grupo.

In [ ]:
df["age_group"] = pd.cut(df.age, [13, 15, 17, 20], labels=["14–15", "16–17", "18–20"])
base_rate = fail.mean()
filas = []
for col in ["family_income", "gender", "urban_rural", "school_type", "education_level", "age_group"]:
    for nivel, idx in df.groupby(col).groups.items():
        n = len(idx); p = fail.loc[idx].mean(); se = (p * (1 - p) / n) ** 0.5
        filas.append({"atributo": col, "grupo": nivel, "estudiantes": n, "% Fail": round(p * 100, 1),
                      "± (IC 95 %)": round(1.96 * se * 100, 1), "vs. total (puntos)": round((p - base_rate) * 100, 1)})
display(pd.DataFrame(filas))

print("\n% de Fail por ingreso familiar y dificultad del examen (¿la brecha de ingreso persiste dentro de cada dificultad?):")
orden = ["Low", "Lower-Middle", "Middle", "Upper-Middle", "High"]
display((df.assign(f=fail).pivot_table(index="family_income", columns="exam_difficulty", values="f", aggfunc="mean")
         .loc[orden] * 100).round(1))

**Hallazgos de equidad:**

- **`family_income` es el único atributo sensible con una diferencia grande:** `Fail` es **75,5 %** con ingreso bajo (`Low`) y **60,5 %** con ingreso alto (`High`), una brecha de **15 puntos** que **persiste dentro de cada nivel de dificultad** del examen (por ejemplo, 59,1 % frente a 40,5 % en exámenes fáciles).
- **`gender`, `urban_rural`, `school_type`, `education_level` y `age`** están entre 68,9 % y 69,8 %, cerca del total (69,2 %). `gender = Other` (69,8 %) tiene solo 2.029 estudiantes y un intervalo de ± 2,0 puntos: **no se distingue del promedio**.
- **Tensión del caso:** `family_income` ayuda a predecir, pero usarlo significaría que el modelo avisa distinto según la riqueza de la familia. Aun si se excluye, **otras variables pueden actuar como sustituto**, así que se audita el *recall*, la precisión y el NPV de `Fail` por grupo, y se mide **cuánto se pierde al excluirla**.
- Es la decisión del grupo y de la institución, no del EDA.

## 8. Tabla de decisiones (entregable del EDA)

Cada una con **cifra → acción → motivo**. Se copia a `reports/01_eda.md` y a `especificaciones/02-eda.md`. Son **propuestas**: el grupo debe revisarlas y aprobarlas.

| # | Hallazgo (con cifra) | Decisión | Motivo |
|---|---|---|---|
| 1 | 6 columnas contienen la respuesta o son del examen: `exam_score` (AUC 0,00), `pass_status` (la misma respuesta con corte 50: 22.611 `Fail` siempre `Fail`), `performance_grade` (da la respuesta exacta a 78,2 %), `performance_level` (58,0 %), `questions_correct` (correlación 0,975 con la nota), `questions_attempted` | **Crear `pass_70` desde `exam_score` y luego excluir las 6 columnas** de las variables de entrada | Darían un modelo perfecto en papel e inútil al decidir, antes del examen |
| 2 | 7 columnas con vacíos (3,0 %–9,9 %); **41,9 % de las filas (41.940) tiene algún vacío**; `Fail` con vacío 68,3 %–69,4 % vs ≈ 69,2 % sin vacío | **No borrar filas**; imputar **dentro del `Pipeline`** con la mediana o una categoría «Desconocido» | Borrar perdería 41,9 % de los estudiantes, y el vacío no anticipa el resultado |
| 3 | `student_id` único (100.000 ids); 0 duplicados | **Partir por fila, 80/20, estratificado por `pass_70`**, semilla 42; excluir `student_id` | Ningún estudiante cae en dos particiones; se mantiene el 69,2 % de `Fail` en ambas |
| 4 | `Fail` = **69,2 %**: «avisar a todos» acierta 69,2 % y cuesta **0,31** (la mejor línea base); «nunca avisar» cuesta 3,46; 21,8 % está a menos de 5 puntos del corte (53,7 % pierde) | **Evaluar con costo por estudiante, *recall* de `Fail` y NPV (≥ 83,3 %)**, con precisión > 69,2 %; no con exactitud; ajustar el umbral con validación | Con la mayoría reprobada, la exactitud y el *recall* solos se «ganan» avisando a todos |
| 5 | `stress_level` tiene 69,9 % de los valores en 10 y `exam_anxiety_level` 38,6 %; `time_management_score` 7,4 % en 100 | **Conservar sin recortar ni transformar** | No son errores: están truncadas arriba. Los modelos de árboles las manejan bien |
| 6 | `self_study_hours` ≤ `study_hours_per_day` en el 100 % de las filas; sumar las tres columnas de horas da 63 estudiantes con más de 24 h | **No crear `total_study_hours` sumando**; probar `self_study_hours / study_hours_per_day` | Sumar duplicaría horas |
| 7 | Señal por variable: `previous_exam_score` (`Fail` 90,7 % → 41,7 % del quintil más bajo al más alto), `previous_gpa` (89,6 % → 43,1 %), `exam_difficulty` (51,1 % → 85,1 %); 7 numéricas con señal ≈ 0,50 | **Conservar todas como candidatas** y medir en la etapa 4 con validación cruzada; no descartar nada solo por el EDA | El EDA usa todos los datos; la selección de variables se decide con validación |
| 8 | `family_income`: `Fail` 75,5 % (bajo) vs 60,5 % (alto), brecha de 15 puntos que persiste dentro de cada dificultad; los demás atributos sensibles 68,9 %–69,8 % | **No usar `family_income` ni `gender` como predictores por defecto**; auditar *recall*, precisión y NPV por grupo; medir cuánto cuesta excluirla | El modelo no debe avisar distinto según la riqueza de la familia; puede haber variables sustitutas |
| 9 | `exam_difficulty` es la mayor señal categórica y `time_management_score` / `exam_anxiety_level` se miden en torno al examen | **Medir con y sin** estas 3 columnas; **confirmar con la institución cuándo se conocen** | Si no se conocen antes del examen, serían fuga |

## Entregable de esta etapa

- [ ] Este notebook ejecuta de principio a fin.
- [ ] La tabla de decisiones tiene **al menos 5** filas con cifra y motivo (tiene 9).
- [ ] `reports/01_eda.md` y `especificaciones/02-eda.md` están actualizados con esta tabla y con el corte de 70.
- [ ] Declaré mi **unidad de análisis** por escrito (sección 1).
- [ ] El grupo confirmó con la institución cuándo se conocen `exam_difficulty`, `time_management_score` y `exam_anxiety_level`.
- [ ] El grupo decidió qué hacer con `family_income` (excluir, o incluir con auditoría).

**Siguiente:** `03b_preprocesamiento_aprueba_o_no.ipynb` (objetivo `pass_70`, limpieza y partición por fila estratificada).